In [1]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr
import pandas as pd
import rpy2.rinterface_lib.sexp as SEXP
from rpy2.robjects.vectors import ListVector
import os
import matplotlib.pyplot as plt
import numpy as np

Error importing in API mode: ImportError('On Windows, cffi mode "ANY" is only "ABI".')
Trying to import in ABI mode.
c:\Python312\Lib\site-packages\rpy2\rinterface\__init__.py:1211: UserWarning: Environment variable "PATH" redefined by R and overriding existing variable. Current: "c:\Python312;c:\Users\millenium\AppData\Roaming\Python\Python312\Scripts;C:\Program Files\Microsoft MPI\Bin\;C:\Program Files\Microsoft SDKs\Azure\CLI2\wbin;C:\Users\millenium\.jabba\jdk\temurin@23.0.2\bin;E:\Program Files\ImageMagick-7.1.1-Q16-HDRI;C:\Python312\Scripts\;C:\Python312\;c:\program files (x86)\jdk/bin;c:\windows\system32;c:\windows;c:\windows\system32\wbem;c:\windows\system32\windowspowershell\v1.0\;c:\windows\system32\openssh\;c:\program files\nvidia corporation\nvidia nvdlisr;e:\program files\git\cmd;c:\program files\docker\docker\resources\bin;c:\users\millenium\appdata\local\microsoft\windowsapps;c:\program files\autofirma\autofirma;c:\users\millenium\appdata\roaming\nvm;c:\program files\nod

In [2]:
parameter_translation_map = {
    "A_viewerServerSelectionDistribution": "SP",
    "B_maxReservation": "MR [B]",
    "B_existingViewerServerSelectionDistribution": "VAms",
    "B_newViewerServerParentSelectionDistribution": "OMSms",
    "B_existingViewerServerSelectionTreeTraversal": "VAdtp",
    "B_newViewerServerParentSelectionTreeTraversal": "OMSdtp",
    "C_maxReservation": "MR [C]",
    "C_reservationType": "RT",
    "C_alreadyLinkedTreeTraversal": "EVST",
    "C_alreadyLinkedDistribution": "EVSP",
    "C_linkedSpaceTreeTraversal": "OVST",
    "C_linkedSpaceDistribution": "OVSP",
    "C_viewerAndLinkFromDistribution": "NVSP",
    "C_publisherServerSelectionDistribution": "PSP",
    "maxReservation": "MR",
}
distribution_cols = ["SP", "ESP", "OSP", "EVSP", "OVSP", "NVSP", "PSP", "VAms", "OMSms"]
tree_traversal_cols = ["EST", "OST", "EVST", "OVST", "VAdtp", "OMSdtp"]
distribution_translation_map = {
    "LOW_LOAD": "LL",
    "HIGH_LOAD": "HL",
    "ROUND_ROBIN": "RR",
    "RANDOM": "RND",
}
tree_traversal_translation_map = {
    "TTB": "TTB",
    "BOTTOM_TO_TOP": "BTT",
    "IGNORE_TREE_LEVEL": "IT",
}
reservation_type_translation_map = {
    "RESERVATION_PER_SERVER": "SER",
    "RESERVATION_PER_SESSION": "SES",
}
alg_a_parameters = ["SP"]
alg_b_parameters = ["MR [B]", "ESP", "OSP", "EST", "OST", "MR"]
alg_c_parameters = ["MR [C]", "RT", "EVST", "EVSP", "OVST", "OVSP", "NVSP", "PSP", "MR"]

In [3]:
def remove_duplicates(configs, training_exps, testing_exps, elites):
    # Some configs are duplicate of others with more experiments in different instances, we want to keep the most informative ones
    duplicate_configs = configs[configs.duplicated(keep=False)]
    # if a row contains a RND parameter, we want to remove those "duplicate" configs from our true duplicates
    duplicate_configs = duplicate_configs[
        ~duplicate_configs.apply(
            lambda row: any(row[col] == "RND" for col in duplicate_configs.columns),
            axis=1,
        )
    ]
    # Group duplicate rows by their values (excluding the index)
    duplicate_groups = {}
    for idx, row in duplicate_configs.iterrows():
        row_key = frozenset(
            (col, str(row[col]) if pd.notna(row[col]) else "NaN")
            for col in duplicate_configs.columns
        )
        duplicate_groups.setdefault(row_key, []).append(idx)

    # Keep only groups with more than one entry
    duplicate_indices_tuples = [
        tuple(indices) for indices in duplicate_groups.values() if len(indices) > 1
    ]

    best_cols = []
    for indices in duplicate_indices_tuples:
        cols = training_exps[list(indices)]
        # Prefer any index present in testing_exps
        if testing_exps.columns.isin(indices).any():
            best_col = next((i for i in indices if i in testing_exps.columns), None)
        else:
            non_na_counts = cols.notna().sum()
            best_col = non_na_counts.idxmax()
        best_cols.append(best_col)
        # Fill gaps in the chosen column from the others
        for other in indices:
            if other == best_col:
                continue
            for row in training_exps.index:
                if pd.isna(training_exps.at[row, best_col]) and pd.notna(
                    training_exps.at[row, other]
                ):
                    training_exps.at[row, best_col] = training_exps.at[row, other]

    # Update elites: replace any dropped duplicate in elites with its kept best_col
    updated_elites = list(elites)
    for indices, best_col in zip(duplicate_indices_tuples, best_cols):
        for dup in indices:
            if dup != best_col and dup in updated_elites:
                updated_elites[updated_elites.index(dup)] = best_col

    # Drop the non-best duplicates
    to_drop = [
        idx for grp in duplicate_indices_tuples for idx in grp if idx not in best_cols
    ]
    new_configs = configs.drop(index=to_drop)
    new_training_exps = training_exps.drop(columns=to_drop)

    return new_configs, new_training_exps, updated_elites


def calc_dev(full_exps, dev):
    for idx, row in full_exps.iterrows():
        min_value = row.min()
        dev.loc[idx] = row.apply(
            lambda x, minimum_value=min_value: (
                100 * (x - minimum_value) / ((x + minimum_value) / 2)
                if pd.notna(x)
                else x
            )
        )


irace = importr("irace")


def load_irace_elite_configs(file_location):
    ro.r["load"](file_location)
    objects = ro.r["iraceResults"]
    testing = ListVector(objects.rx2("testing"))
    test_exp_matrix = testing.rx2("experiments")
    test_matrix_row_names = list(test_exp_matrix.rownames)
    test_matrix_col_names = list(test_exp_matrix.colnames)
    training_exps = objects.rx2("experiments")
    training_exps_col_names = list(training_exps.colnames)
    with (ro.default_converter + pandas2ri.converter).context():
        instances = irace.get_instanceID_seed_pairs(objects, instances=True)
        test_exp_matrix = pandas2ri.rpy2py(test_exp_matrix)
        test_exp_matrix = pd.DataFrame(
            test_exp_matrix, index=test_matrix_row_names, columns=test_matrix_col_names
        )
        training_exps = pandas2ri.rpy2py(training_exps)
        training_exps = pd.DataFrame(
            training_exps,
            index=instances["instanceID"][:len(training_exps)],
            columns=training_exps_col_names,
        )
        training_exps = training_exps.sort_index()
        configs = objects.rx2("allConfigurations")
        configs = configs.replace(-2147483648, None)
        for col in configs.columns:
            configs[col] = configs[col].apply(
                lambda x: None if isinstance(x, SEXP.NACharacterType) else x
            )
        configs = configs.drop(columns=[".ID.", ".PARENT."])
        configs = configs.rename(columns=parameter_translation_map)
        for col in configs.select_dtypes(include=["float"]):
            if (configs[col] % 1 == 0).all():
                configs[col] = configs[col].astype("Int64")
        for col in configs.columns:
            if col in distribution_cols:
                configs[col] = configs[col].map(distribution_translation_map)
            if col in tree_traversal_cols:
                configs[col] = configs[col].map(tree_traversal_translation_map)
            if col == "RT":
                configs[col] = configs[col].map(reservation_type_translation_map)
        elites = objects.rx2("allElites")[-1]
        elites = [str(e) for e in elites]
        configs, training_exps, elites = remove_duplicates(
            configs, training_exps, test_exp_matrix, elites
        )
        elite_configs = configs.loc[elites]
        elite_configs = elite_configs.dropna(axis=1, how="all")
        for col in elite_configs.select_dtypes(include=["float"]):
            if (elite_configs[col] % 1 == 0).all():
                elite_configs[col] = elite_configs[col].astype("Int64")

        all_exps = pd.concat([training_exps, test_exp_matrix], axis=0)
        # coming from all_exps, for every row, in each cell change the value for the deviation from the best config
        dev = all_exps.copy()
        calc_dev(all_exps, dev)
        return configs, elite_configs, test_exp_matrix, training_exps, all_exps, dev


maps = {
    "All": {
        "configs_map": {},
        "elite_configs_map": {},
        "testing_exps_map": {},
        "training_exps_map": {},
        "full_experiments_map": {},
        "deviation_map": {},
    },
    "A": {
        "configs_map": {},
        "elite_configs_map": {},
        "testing_exps_map": {},
        "training_exps_map": {},
        "full_experiments_map": {},
        "deviation_map": {},
    },
    "B": {
        "configs_map": {},
        "elite_configs_map": {},
        "testing_exps_map": {},
        "training_exps_map": {},
        "full_experiments_map": {},
        "deviation_map": {},
    },
}
for root, _, files in os.walk("irace_results"):
    for file in files:
        if file.endswith("irace.Rdata"):
            file_path = os.path.join(root, file)
            dir_path = os.path.dirname(file_path)
            splitted = dir_path.split(os.sep)
            alg = "All"
            if "A" in file_path:
                alg = "A"
                budget = 1000
                instance_type = splitted[3]
                server_capacity = splitted[4]
            elif "B" in file_path:
                alg = "B"
                budget = 5000
                instance_type = splitted[3]
                server_capacity = splitted[4]
            else:
                budget = splitted[1].split("_")[0]
                instance_type = splitted[2]
                server_capacity = splitted[3]
            configs_map = maps[alg]["configs_map"]
            elite_configs_map = maps[alg]["elite_configs_map"]
            testing_exps_map = maps[alg]["testing_exps_map"]
            training_exps_map = maps[alg]["training_exps_map"]
            full_experiments_map = maps[alg]["full_experiments_map"]
            deviation_map = maps[alg]["deviation_map"]
            try:
                configs, elite_configs, testing_exps, full_exps, all_exps, dev = (
                    load_irace_elite_configs(file_path)
                )
                # Ensure 'small' comes before 'medium' in the map
                if instance_type not in configs_map:
                    configs_map[instance_type] = {}
                configs_map[instance_type][server_capacity] = configs
                if instance_type not in elite_configs_map:
                    elite_configs_map[instance_type] = {}
                elite_configs_map[instance_type][server_capacity] = elite_configs
                if instance_type not in testing_exps_map:
                    testing_exps_map[instance_type] = {}
                testing_exps_map[instance_type][server_capacity] = testing_exps
                if instance_type not in training_exps_map:
                    training_exps_map[instance_type] = {}
                training_exps_map[instance_type][server_capacity] = full_exps
                if instance_type not in full_experiments_map:
                    full_experiments_map[instance_type] = {}
                full_experiments_map[instance_type][server_capacity] = all_exps
                if instance_type not in deviation_map:
                    deviation_map[instance_type] = {}
                deviation_map[instance_type][server_capacity] = dev
            except Exception as e:
                print(f"Failed to process {file_path}: {e}")


# Reorder the maps so 'small' comes first, then 'medium', then 'big' and server capacities are sorted numerically
def reorder_instance_types_and_capacities(d):
    ordered = {}
    for instance in ["small", "medium", "big"]:
        if instance in d:
            # Sort server capacities numerically
            capacities = d[instance]
            ordered_capacities = dict(
                sorted(capacities.items(), key=lambda x: int(x[0]))
            )
            ordered[instance] = ordered_capacities
    for instance in d:
        if instance not in ordered:
            capacities = d[instance]
            ordered_capacities = dict(
                sorted(capacities.items(), key=lambda x: int(x[0]))
            )
            ordered[instance] = ordered_capacities
    return ordered


for alg in maps:
    configs_map = maps[alg]["configs_map"]
    elite_configs_map = maps[alg]["elite_configs_map"]
    testing_exps_map = maps[alg]["testing_exps_map"]
    training_exps_map = maps[alg]["training_exps_map"]
    full_experiments_map = maps[alg]["full_experiments_map"]
    deviation_map = maps[alg]["deviation_map"]
    maps[alg]["configs_map"] = reorder_instance_types_and_capacities(configs_map)
    maps[alg]["elite_configs_map"] = reorder_instance_types_and_capacities(
        elite_configs_map
    )
    maps[alg]["testing_exps_map"] = reorder_instance_types_and_capacities(
        testing_exps_map
    )
    maps[alg]["training_exps_map"] = reorder_instance_types_and_capacities(
        training_exps_map
    )
    maps[alg]["full_experiments_map"] = reorder_instance_types_and_capacities(
        full_experiments_map
    )
    maps[alg]["deviation_map"] = reorder_instance_types_and_capacities(deviation_map)

In [4]:
def config_summary_str(config_row, subindex=None, alg=None):
    algorithm = alg
    if algorithm is None:
        algorithm = config_row["algorithm"]
    if subindex is not None:
        result = f"{algorithm}<sub>{subindex}</sub>("
    else:
        result = f"{algorithm}("
    for col in config_row.index:
        if col == "algorithm":
            continue
        value = config_row[col]
        if pd.isna(value):
            continue
        if isinstance(value, (float, np.floating)):
            value = int(value)
        key = col
        if "MR" in col:
            key = "MR"
        result += f"{key}={value}, "
    return result[:-2] + ")"

summaries_elites = {}
summaries_elites_A = {}
summaries_elites_B = {}

def summarize_elite_configs(alg="All", df=summaries_elites):
    for instance_type, server_capacities in maps[alg]["full_experiments_map"].items():
        for server_capacity, full_exps in server_capacities.items():
            elite_configs = maps[alg]["elite_configs_map"][instance_type][server_capacity]
            elite_index = elite_configs.index
            devs = maps[alg]["deviation_map"][instance_type][server_capacity]
            # mean deviation
            mean_devs = []
            for idx in elite_index:
                mean_devs.append(devs[idx].mean())
            mean_devs = np.array(mean_devs)
            # mean rank
            mean_rank = [0] * len(elite_index)
            best_exps = full_exps[elite_index]
            for _, row in best_exps.iterrows():
                ranks = row.rank(method="min")
                for i, idx in enumerate(elite_index):
                    rank = ranks[idx]
                    if pd.notna(rank):
                        mean_rank[i] += rank
            mean_rank = np.array(mean_rank) / len(best_exps)

            configs = maps[alg]["configs_map"][instance_type][server_capacity]
            summary_config = [
                config_summary_str(configs.loc[idx], subindex=i + 1, alg=alg if alg != "All" else None)
                for i, idx in enumerate(elite_index)
            ]
            if instance_type not in df:
                df[instance_type] = {}
            df[instance_type][server_capacity] = {
                "Configuration": summary_config,
                "Config ID": elite_index,
                # "# Instances run on": n_instances,
                "Mean RPD": mean_devs,
                # "# Best": n_best,
                "Mean Rank": mean_rank,
            }

summarize_elite_configs()
summarize_elite_configs(alg="A", df=summaries_elites_A)
summarize_elite_configs(alg="B", df=summaries_elites_B)

def display_summary_df(df):
    summary = pd.DataFrame(
        columns=[
            "Instance type",
            "Server capacity",
            "Config ID",
            "Configuration",
            # "# Instances run on",
            "Mean RPD",
            # "# Best",
            "Mean Rank",
        ]
    )
    for instance_type, server_capacities in df.items():
        for server_capacity, summary_data in server_capacities.items():
            n = len(summary_data["Configuration"])
            df = pd.DataFrame(
                {
                    "Instance type": [instance_type] * n,
                    "Server capacity": [server_capacity] * n,
                    **summary_data,
                }
            )
            summary = pd.concat([summary, df], ignore_index=True)
    display(summary.style.hide(axis="index"))
display_summary_df(summaries_elites)
display_summary_df(summaries_elites_A)
display_summary_df(summaries_elites_B)

C:\Users\millenium\AppData\Local\Temp\ipykernel_136892\4232430307.py:92: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  summary = pd.concat([summary, df], ignore_index=True)


Instance type,Server capacity,Config ID,Configuration,Mean RPD,Mean Rank
small,50,2094,"C1(MR=2, RT=SER, EVSP=LL, OVST=BTT, OVSP=LL, NVSP=HL, PSP=RR)",2.875588,2.800000
small,50,3304,"C2(MR=2, RT=SER, EVSP=LL, OVST=BTT, OVSP=RND, NVSP=HL, PSP=RND)",2.838135,2.500000
small,50,4037,"C3(MR=2, RT=SER, EVSP=LL, OVST=BTT, OVSP=RND, NVSP=LL, PSP=RND)",2.838135,2.500000
small,50,7035,"C4(MR=2, RT=SER, EVSP=LL, OVST=BTT, OVSP=RND, NVSP=HL, PSP=RR)",2.862590,3.100000
small,50,11581,"C5(MR=2, RT=SER, EVSP=LL, OVST=BTT, OVSP=HL, NVSP=HL, PSP=RR)",2.757602,2.925000
small,150,13764,"C1(MR=3, RT=SER, EVSP=LL, OVST=IT, OVSP=RND, NVSP=LL, PSP=RR)",2.534669,3.000000
small,150,11479,"C2(MR=3, RT=SER, EVSP=LL, OVST=IT, OVSP=RR, NVSP=HL, PSP=RR)",2.521540,2.675000
small,150,10236,"C3(MR=3, RT=SER, EVSP=LL, OVSP=RND, NVSP=LL, PSP=RR)",2.672769,3.275000
small,150,4085,"C4(MR=3, RT=SER, EVSP=LL, OVST=IT, OVSP=RND, NVSP=LL, PSP=LL)",2.641216,2.525000
small,150,5220,"C5(MR=3, RT=SER, EVSP=LL, OVST=IT, OVSP=RND, NVSP=HL, PSP=LL)",2.641216,2.525000


C:\Users\millenium\AppData\Local\Temp\ipykernel_136892\4232430307.py:92: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  summary = pd.concat([summary, df], ignore_index=True)


Instance type,Server capacity,Config ID,Configuration,Mean RPD,Mean Rank
small,50,6,A1(SP=HL),0.000000,1.000000
small,150,2,A1(SP=HL),0.000000,1.000000
small,650,2,A1(SP=HL),0.000000,1.000000
small,1000,4,A1(SP=HL),0.000000,1.000000
medium,50,1,A1(SP=HL),0.000000,1.000000
medium,150,4,A1(SP=HL),0.000000,1.000000
medium,650,1,A1(SP=HL),0.000000,1.000000
medium,1000,2,A1(SP=HL),0.000000,1.000000
big,50,1,A1(SP=RND),0.000000,1.000000
big,150,1,A1(SP=LL),0.000000,1.000000


C:\Users\millenium\AppData\Local\Temp\ipykernel_136892\4232430307.py:92: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  summary = pd.concat([summary, df], ignore_index=True)


Instance type,Server capacity,Config ID,Configuration,Mean RPD,Mean Rank
small,50,14,"B1(MR=1, VAms=HL, OMSms=LL)",0.841161,1.000000
small,50,93,"B2(MR=1, VAdtp=BTT, VAms=RR, OMSdtp=IT, OMSms=RR)",0.841161,1.000000
small,50,144,"B3(MR=1, VAms=RND, OMSdtp=IT, OMSms=HL)",0.841161,1.000000
small,50,176,"B4(MR=1, VAms=HL, OMSms=HL)",0.841161,1.000000
small,150,55,"B1(MR=1, VAms=RND, OMSms=HL)",1.202408,1.525000
small,150,218,"B2(MR=1, VAms=LL, OMSms=LL)",1.202408,1.525000
small,150,832,"B3(MR=1, VAms=HL, OMSms=RR)",1.202408,1.525000
small,150,716,"B4(MR=2, VAms=HL, OMSms=HL)",0.383238,2.425000
small,650,86,"B1(MR=1, VAms=LL, OMSdtp=BTT, OMSms=RND)",0.176598,1.000000
small,650,499,"B2(MR=1, VAdtp=BTT, VAms=RR, OMSdtp=BTT, OMSms=RND)",0.176598,1.000000


In [5]:
# The first elite config is the best one considered by irace, so we run each best elite config on all combinations of instance types and server capacities
# (running them only on test instances)

best_elites = {
    "C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, OVSP=LL, NVSP=HL, PSP=RR)": [
        ("small", "50"),
        ("medium", "50"),
    ],
    "C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, OVSP=RND, NVSP=LL, PSP=RR)": (
        "small",
        "150",
    ),
    "C1(MR=2, RT=SES, EVST=TTB, EVSP=LL, OVST=TTB, OVSP=LL, NVSP=LL, PSP=LL)": (
        "small",
        "650",
    ),
    "C1(MR=9, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, OVSP=LL, NVSP=HL, PSP=LL)": (
        "small",
        "1000",
    ),
    "C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, OVSP=HL, NVSP=LL, PSP=RR)": (
        "medium",
        "150",
    ),
    "C1(MR=6, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, OVSP=LL, NVSP=HL, PSP=RND)": (
        "medium",
        "650",
    ),
    "C1(MR=9, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, OVSP=LL, NVSP=LL, PSP=LL)": (
        "medium",
        "1000",
    ),
    "C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, OVSP=RR, NVSP=LL, PSP=RND)": (
        "big",
        "50",
    ),
    "C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, OVSP=RR, NVSP=LL, PSP=LL)": (
        "big",
        "150",
    ),
    "C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, OVSP=RR, NVSP=HL, PSP=RR)": (
        "big",
        "650",
    ),
    "C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, OVSP=RND, NVSP=HL, PSP=RND)": (
        "big",
        "1000",
    )
}

best_elites_b = {
    "B1(MR=1, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms=LL)": ("small", "50"),
    "B1(MR=1, VAdtp=TTB, VAms=RND, OMSdtp=TTB, OMSms=HL)": ("small", "150"),
    "B1(MR=1, VAdtp=TTB, VAms=LL, OMSdtp=BTT, OMSms=RND)": ("small", "650"),
    "B1(MR=1, VAdtp=TTB, VAms=LL, OMSdtp=BTT, OMSms=RR)": ("small", "1000"),
    "B1(MR=2, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms=LL)": [("medium", "50"), ("medium", "650"), ("big", "150")],
    "B1(MR=2, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms=HL)": [("medium", "150"), ("big", "50")],
    "B1(MR=3, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms=LL)": ("medium", "1000"),
    "B1(MR=3, VAdtp=TTB, VAms=HL, OMSdtp=IT, OMSms=HL)": ("big", "650"),
    "B1(MR=1, VAdtp=TTB, VAms=RND, OMSdtp=BTT, OMSms=RR)": ("big", "1000"),
}

best_elites_a = {
    "A1(SP=HL)": [("small", "50"), ("small", "150"), ("small", "650"), ("small", "1000"), ("medium", "50"), ("medium", "150"), ("medium", "650"), ("medium", "1000"), ("big", "50"), ("big", "150"), ("big", "650"), ("big", "1000")]
}

def convert_best_elites_to_df(be):
# convert best_elites to a dataframe
    best_elites_df = pd.DataFrame(
        columns=["Instance type", "Server capacity", "Configuration"]
    )

    for config, instances in be.items():
        if isinstance(instances, tuple):
            instances = [instances]
        for instance in instances:
            best_elites_df = pd.concat(
                [
                    best_elites_df,
                    pd.DataFrame(
                        {
                            "Instance type": [instance[0]],
                            "Server capacity": [int(instance[1])],
                            "Configuration": [config],
                        }
                    ),
                ],
                ignore_index=True,
            )

    best_elites_df = best_elites_df.sort_values(by=["Instance type", "Server capacity"], ascending=[False, True])
    return best_elites_df

best_elites_df = convert_best_elites_to_df(best_elites)
best_elites_df_b = convert_best_elites_to_df(best_elites_b)
best_elites_df_a = convert_best_elites_to_df(best_elites_a)
display(best_elites_df)
best_elites_df.to_latex(
    "best_elites.tex",
    index=False,
    caption="Best Elite Configurations for Each Instance Type and Server Capacity",
)
display(best_elites_df_b)
best_elites_df_b.to_latex(
    "best_elites_b.tex",
    index=False,
    caption="Best Elite Configurations for Each Instance Type and Server Capacity (B)",
)
display(best_elites_df_a)

,Instance type,Server capacity,Configuration
0,small,50,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ..."
2,small,150,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, O..."
3,small,650,"C1(MR=2, RT=SES, EVST=TTB, EVSP=LL, OVST=TTB, ..."
4,small,1000,"C1(MR=9, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, O..."
1,medium,50,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ..."
5,medium,150,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ..."
6,medium,650,"C1(MR=6, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ..."
7,medium,1000,"C1(MR=9, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ..."
8,big,50,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ..."
9,big,150,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ..."


,Instance type,Server capacity,Configuration
0,small,50,"B1(MR=1, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms..."
1,small,150,"B1(MR=1, VAdtp=TTB, VAms=RND, OMSdtp=TTB, OMSm..."
2,small,650,"B1(MR=1, VAdtp=TTB, VAms=LL, OMSdtp=BTT, OMSms..."
3,small,1000,"B1(MR=1, VAdtp=TTB, VAms=LL, OMSdtp=BTT, OMSms..."
4,medium,50,"B1(MR=2, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms..."
7,medium,150,"B1(MR=2, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms..."
5,medium,650,"B1(MR=2, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms..."
9,medium,1000,"B1(MR=3, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms..."
8,big,50,"B1(MR=2, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms..."
6,big,150,"B1(MR=2, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms..."


,Instance type,Server capacity,Configuration
0,small,50,A1(SP=HL)
1,small,150,A1(SP=HL)
2,small,650,A1(SP=HL)
3,small,1000,A1(SP=HL)
4,medium,50,A1(SP=HL)
5,medium,150,A1(SP=HL)
6,medium,650,A1(SP=HL)
7,medium,1000,A1(SP=HL)
8,big,50,A1(SP=HL)
9,big,150,A1(SP=HL)


In [6]:
best_elite_runs = pd.DataFrame(
    columns=[
        "Instance type",
        "Server capacity",
        # "Was best elite",
        "Instance",
        "Configuration",
        "Cost",
        # "Time"
    ]
)
best_elite_runs_b = best_elite_runs.copy()
best_elite_runs_a = best_elite_runs.copy()
def parse_log(file, instance_type, be, ber):
    with open(file, "r") as f:
        df_row = {
            "Instance type": instance_type,
            "Server capacity": None,
            "Was best elite": None,
            "Configuration": None,
            "Instance": None,
            "Cost": None,
        }
        skip_next = False
        for line in f:
            if skip_next:
                skip_next = False
                continue
            if "Output for" in line:
                # Find substring CAPACITY= and read the number after it, preceeding the ,
                server_capacity = line.split("CAPACITY=")[1].split(",")[0]
                instance = line.split(",")[1].split("=")[1]
                real_instance = str(int(instance) - 29) + "t"
                config = line.split("args=")[1].strip()
                best_config = be.get(config, None)
                if best_config is None:
                    skip_next = True
                    continue
                if isinstance(best_config, list):
                    for best in best_config:
                        df_row["Was best elite"] = "F"
                        if best[0] == instance_type and best[1] == server_capacity:
                            df_row["Was best elite"] = "T"
                            break
                else:
                    if (
                        best_config[0] == instance_type
                        and best_config[1] == server_capacity
                    ):
                        df_row["Was best elite"] = "T"
                    else:
                        df_row["Was best elite"] = "F"
                df_row["Server capacity"] = int(server_capacity)
                df_row["Instance"] = real_instance
                df_row["Configuration"] = config
            else:
                if "Error" in line:
                    df_row["Cost"] = "E"
                else:
                    cost = float(line.split(" ")[0])
                    df_row["Cost"] = cost
                ber.loc[len(ber)] = df_row


parse_log("test_elite_configs/a_small.log", "small", best_elites_a, best_elite_runs_a)
parse_log("test_elite_configs/a_medium.log", "medium", best_elites_a, best_elite_runs_a)
parse_log("test_elite_configs/a_big.log", "big", best_elites_a, best_elite_runs_a)
parse_log("test_elite_configs/b_small.log", "small", best_elites_b, best_elite_runs_b)
parse_log("test_elite_configs/b_medium.log", "medium", best_elites_b, best_elite_runs_b)
parse_log("test_elite_configs/b_big.log", "big", best_elites_b, best_elite_runs_b)
parse_log("test_elite_configs/small.log", "small", best_elites, best_elite_runs)
parse_log("test_elite_configs/medium.log", "medium", best_elites, best_elite_runs)
parse_log("test_elite_configs/big.log", "big", best_elites, best_elite_runs)
display(
    best_elite_runs.sort_values(
        by=["Instance type", "Server capacity", "Instance"],
        ascending=[False, True, True],
    )
)
display(
    best_elite_runs_b.sort_values(
        by=["Instance type", "Server capacity", "Instance"],
        ascending=[False, True, True],
    )
)
display(
    best_elite_runs_a.sort_values(
        by=["Instance type", "Server capacity", "Instance"],
        ascending=[False, True, True],
    )
)

,Instance type,Server capacity,Instance,Configuration,Cost
9,small,50,10t,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",3.292536e+06
49,small,50,10t,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, O...",3.234938e+06
89,small,50,10t,"C1(MR=2, RT=SES, EVST=TTB, EVSP=LL, OVST=TTB, ...",3.290405e+06
129,small,50,10t,"C1(MR=9, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, O...",3.580695e+06
169,small,50,10t,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",3.428797e+06
...,...,...,...,...,...
1158,big,1000,9t,"C1(MR=9, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",2.190575e+07
1198,big,1000,9t,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",2.223317e+07
1238,big,1000,9t,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",2.243562e+07
1278,big,1000,9t,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",2.181769e+07


,Instance type,Server capacity,Instance,Configuration,Cost
9,small,50,10t,"B1(MR=1, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms...",3.601821e+06
49,small,50,10t,"B1(MR=1, VAdtp=TTB, VAms=RND, OMSdtp=TTB, OMSm...",3.601821e+06
89,small,50,10t,"B1(MR=1, VAdtp=TTB, VAms=LL, OMSdtp=BTT, OMSms...",3.601821e+06
129,small,50,10t,"B1(MR=1, VAdtp=TTB, VAms=LL, OMSdtp=BTT, OMSms...",3.601821e+06
169,small,50,10t,"B1(MR=2, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms...",3.224463e+06
...,...,...,...,...,...
918,big,1000,9t,"B1(MR=2, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms...",2.584202e+07
958,big,1000,9t,"B1(MR=2, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms...",2.584202e+07
998,big,1000,9t,"B1(MR=3, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms...",2.585724e+07
1038,big,1000,9t,"B1(MR=3, VAdtp=TTB, VAms=HL, OMSdtp=IT, OMSms=HL)",2.585679e+07


,Instance type,Server capacity,Instance,Configuration,Cost
9,small,50,10t,A1(SP=HL),E
0,small,50,1t,A1(SP=HL),E
1,small,50,2t,A1(SP=HL),E
2,small,50,3t,A1(SP=HL),E
3,small,50,4t,A1(SP=HL),E
...,...,...,...,...,...
114,big,1000,5t,A1(SP=HL),15421104.102
115,big,1000,6t,A1(SP=HL),34428240.111
116,big,1000,7t,A1(SP=HL),26048557.485
117,big,1000,8t,A1(SP=HL),25752812.767


In [7]:
# For each combination of instance type, server capacity and instance in best_elite_runs, calculate the RPD of all configurations
def calc_stats_elites(best_elite_runs):
    best_elite_stats = best_elite_runs.copy()
    instance_types = best_elite_stats["Instance type"].unique()
    server_capacities = best_elite_stats["Server capacity"].unique()
    instances = best_elite_stats["Instance"].unique()
    for instance_type in instance_types:
        for server_capacity in server_capacities:
            for instance in instances:
                best_elites = best_elite_stats[
                    (best_elite_stats["Instance type"] == instance_type)
                    & (best_elite_stats["Server capacity"] == server_capacity)
                    & (best_elite_stats["Instance"] == instance)
                ]
                ranks = best_elites["Cost"].rank(method="min")
                best_elite_stats.loc[
                    (best_elite_stats["Instance type"] == instance_type)
                    & (best_elite_stats["Server capacity"] == server_capacity)
                    & (best_elite_stats["Instance"] == instance),
                    "Rank",
                ] = ranks
                best_elite_run = best_elites.sort_values(by="Cost").iloc[0]
                best_elite_cost = best_elite_run["Cost"]
                for _, row in best_elites.iterrows():
                    config = row["Configuration"]
                    cost = row["Cost"]
                    rpd = 100 * abs((cost - best_elite_cost) / best_elite_cost)
                    best_elite_stats.loc[
                        (best_elite_stats["Instance type"] == instance_type)
                        & (best_elite_stats["Server capacity"] == server_capacity)
                        & (best_elite_stats["Instance"] == instance)
                        & (best_elite_stats["Configuration"] == config)
                        & (best_elite_stats["Cost"] == cost),
                        "RPD",
                    ] = rpd.round(3)
                    # best_elite_stats.loc[
                    #     (best_elite_stats["Instance type"] == instance_type)
                    #     & (best_elite_stats["Server capacity"] == server_capacity)
                    #     & (best_elite_stats["Instance"] == instance)
                    #     & (best_elite_stats["Configuration"] == config)
                    #     & (best_elite_stats["Cost"] == cost),
                    #     "Is best",
                    # ] = (
                    #     cost == best_elite_cost
                    # )
    return best_elite_stats

best_elite_stats = calc_stats_elites(best_elite_runs)
display(
    best_elite_stats.sort_values(
        by=["Instance type", "Server capacity", "Instance"],
        ascending=[False, True, True],
    )
)
best_elite_stats_b = calc_stats_elites(best_elite_runs_b)
display(
    best_elite_stats_b.sort_values(
        by=["Instance type", "Server capacity", "Instance"],
        ascending=[False, True, True],
    )
)

,Instance type,Server capacity,Instance,Configuration,Cost,Rank,RPD
9,small,50,10t,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",3.292536e+06,5.0,1.780
49,small,50,10t,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, O...",3.234938e+06,1.0,0.000
89,small,50,10t,"C1(MR=2, RT=SES, EVST=TTB, EVSP=LL, OVST=TTB, ...",3.290405e+06,4.0,1.715
129,small,50,10t,"C1(MR=9, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, O...",3.580695e+06,10.0,10.688
169,small,50,10t,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",3.428797e+06,9.0,5.993
...,...,...,...,...,...,...,...
1158,big,1000,9t,"C1(MR=9, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",2.190575e+07,6.0,1.819
1198,big,1000,9t,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",2.223317e+07,8.0,3.341
1238,big,1000,9t,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",2.243562e+07,10.0,4.282
1278,big,1000,9t,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",2.181769e+07,5.0,1.410


,Instance type,Server capacity,Instance,Configuration,Cost,Rank,RPD
9,small,50,10t,"B1(MR=1, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms...",3.601821e+06,5.0,11.703
49,small,50,10t,"B1(MR=1, VAdtp=TTB, VAms=RND, OMSdtp=TTB, OMSm...",3.601821e+06,5.0,11.703
89,small,50,10t,"B1(MR=1, VAdtp=TTB, VAms=LL, OMSdtp=BTT, OMSms...",3.601821e+06,5.0,11.703
129,small,50,10t,"B1(MR=1, VAdtp=TTB, VAms=LL, OMSdtp=BTT, OMSms...",3.601821e+06,5.0,11.703
169,small,50,10t,"B1(MR=2, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms...",3.224463e+06,1.0,0.000
...,...,...,...,...,...,...,...
918,big,1000,9t,"B1(MR=2, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms...",2.584202e+07,6.0,0.083
958,big,1000,9t,"B1(MR=2, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms...",2.584202e+07,6.0,0.083
998,big,1000,9t,"B1(MR=3, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms...",2.585724e+07,9.0,0.142
1038,big,1000,9t,"B1(MR=3, VAdtp=TTB, VAms=HL, OMSdtp=IT, OMSms=HL)",2.585679e+07,8.0,0.140


In [8]:
def group_alt_results(stats, group_array, sort_by=None, ascending=None):
    avgs = stats.groupby(group_array)["RPD"].mean().round(3).reset_index()
    avgs = avgs.rename(columns={"RPD": "Mean RPD"})

    std = stats.groupby(group_array)["RPD"].std().round(3).reset_index()
    avgs["Mean RPD"] = avgs["Mean RPD"].astype(str) + " (" + std["RPD"].astype(str) + ")"
    avgs = avgs.rename(columns={"Mean RPD": "Mean RPD (SD)"})

    # max_rpd = stats.groupby(group_array)["RPD"].max().round(3).reset_index()
    # avgs = avgs.merge(
    #     max_rpd,
    #     on=group_array,
    # )
    # avgs = avgs.rename(columns={"RPD": "Max RPD"})

    # q1 = (
    #     stats.groupby(group_array)["RPD"]
    #     .quantile(0.25)
    #     .round(3)
    #     .reset_index()
    # )
    # avgs = avgs.merge(q1, on=group_array)
    # avgs = avgs.rename(columns={"RPD": "Q1 RPD"})
    # median = stats.groupby(group_array)["RPD"].median().round(3).reset_index()
    # avgs = avgs.merge(
    #     median,
    #     on=group_array,
    # )
    # avgs = avgs.rename(columns={"RPD": "Median RPD"})
    # q3 = (
    #     stats.groupby(group_array)["RPD"]
    #     .quantile(0.75)
    #     .round(3)
    #     .reset_index()
    # )
    # avgs = avgs.merge(q3, on=group_array)
    # avgs = avgs.rename(columns={"RPD": "Q3 RPD"})

    # best_counts = (
    #     stats.groupby(group_array)["Is best"]
    #     .apply(lambda x: x.eq(True).sum())
    #     .reset_index(name="# Best")
    # )
    # avgs = avgs.merge(
    #     best_counts,
    #     on=group_array,
    # )
    avgs["Mean Rank"] = (
        stats.groupby(group_array)["Rank"]
        .mean()
        .round(3)
        .reset_index(drop=True)
    )

    return avgs.sort_values(by=sort_by, ascending=ascending)


# convert to tex table
def to_latex_table(df, filename):
    with open(filename, "w") as f:
        f.write(
            df.to_latex(
                index=False,
                float_format="%.3f",
                column_format="l" + "c" * (len(df.columns) - 1),
                escape=False,
            )
        )

In [9]:
avgs_all = group_alt_results(
    best_elite_stats,
    ["Configuration"],
    sort_by=["Configuration"],
    ascending=[True],
)
display(avgs_all)
to_latex_table(avgs_all, filename="alt_results_all.tex")
avgs_b = group_alt_results(
    best_elite_stats_b,
    ["Configuration"],
    sort_by=["Configuration"],
    ascending=[True],
)
display(avgs_b)
to_latex_table(avgs_b, filename="alt_results_b.tex")

,Configuration,Mean RPD (SD),Mean Rank
0,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",3.894 (4.792),6.067
1,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",3.844 (4.835),5.958
2,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",4.131 (5.006),6.258
3,"C1(MR=2, RT=SES, EVST=TTB, EVSP=LL, OVST=TTB, ...",6.34 (5.768),7.475
4,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",3.231 (4.474),5.317
5,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, O...",2.609 (3.028),4.900
6,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",2.455 (2.406),5.083
7,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",2.688 (2.725),5.183
8,"C1(MR=6, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",3.533 (3.735),5.267
9,"C1(MR=9, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, O...",6.832 (6.94),7.550


,Configuration,Mean RPD (SD),Mean Rank
0,"B1(MR=1, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms...",0.978 (1.744),3.567
1,"B1(MR=1, VAdtp=TTB, VAms=LL, OMSdtp=BTT, OMSms...",0.978 (1.744),3.567
2,"B1(MR=1, VAdtp=TTB, VAms=LL, OMSdtp=BTT, OMSms...",0.978 (1.744),3.567
3,"B1(MR=1, VAdtp=TTB, VAms=RND, OMSdtp=BTT, OMSm...",0.978 (1.744),3.567
4,"B1(MR=1, VAdtp=TTB, VAms=RND, OMSdtp=TTB, OMSm...",0.978 (1.744),3.567
5,"B1(MR=2, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms...",0.107 (0.337),2.533
6,"B1(MR=2, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms...",0.107 (0.337),2.533
7,"B1(MR=3, VAdtp=TTB, VAms=HL, OMSdtp=IT, OMSms=HL)",0.69 (0.848),5.458
8,"B1(MR=3, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms...",0.682 (0.844),5.200


In [10]:
avgs_instance = group_alt_results(
    best_elite_stats,
    ["Instance type", "Configuration"],
    sort_by=["Instance type", "Configuration"],
    ascending=[False, True],
)
display(avgs_instance)
to_latex_table(avgs_instance, filename="alt_results_instance.tex")
avgs_instance_b = group_alt_results(
    best_elite_stats_b,
    ["Instance type", "Configuration"],
    sort_by=["Instance type", "Configuration"],
    ascending=[False, True],
)
display(avgs_instance_b)
to_latex_table(avgs_instance_b, filename="alt_results_instance_b.tex")

,Instance type,Configuration,Mean RPD (SD),Mean Rank
22,small,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",6.183 (6.414),6.550
23,small,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",5.969 (6.451),6.375
24,small,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",6.725 (6.62),6.525
25,small,"C1(MR=2, RT=SES, EVST=TTB, EVSP=LL, OVST=TTB, ...",3.37 (2.996),4.550
26,small,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",6.545 (6.147),7.675
27,small,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, O...",4.523 (4.357),5.500
28,small,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",3.937 (3.268),5.300
29,small,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",4.579 (3.671),5.825
30,small,"C1(MR=6, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",3.909 (3.386),4.975
31,small,"C1(MR=9, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, O...",6.263 (5.839),6.550


,Instance type,Configuration,Mean RPD (SD),Mean Rank
18,small,"B1(MR=1, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms...",1.312 (2.558),2.700
19,small,"B1(MR=1, VAdtp=TTB, VAms=LL, OMSdtp=BTT, OMSms...",1.312 (2.558),2.700
20,small,"B1(MR=1, VAdtp=TTB, VAms=LL, OMSdtp=BTT, OMSms...",1.312 (2.558),2.700
21,small,"B1(MR=1, VAdtp=TTB, VAms=RND, OMSdtp=BTT, OMSm...",1.312 (2.558),2.700
22,small,"B1(MR=1, VAdtp=TTB, VAms=RND, OMSdtp=TTB, OMSm...",1.312 (2.558),2.700
23,small,"B1(MR=2, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms...",0.241 (0.538),3.775
24,small,"B1(MR=2, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms...",0.241 (0.538),3.775
25,small,"B1(MR=3, VAdtp=TTB, VAms=HL, OMSdtp=IT, OMSms=HL)",0.739 (1.031),6.075
26,small,"B1(MR=3, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms...",0.728 (1.032),6.125
9,medium,"B1(MR=1, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms...",1.101 (1.473),4.150


In [11]:
avgs_instance = group_alt_results(
    best_elite_stats,
    ["Server capacity", "Configuration"],
    sort_by=["Server capacity", "Configuration"],
    ascending=[True, True],
)
display(avgs_instance)
to_latex_table(avgs_instance, filename="alt_results_server.tex")

,Server capacity,Configuration,Mean RPD (SD),Mean Rank
0,50,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",1.531 (4.571),3.233
1,50,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",1.544 (4.807),3.367
2,50,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",1.485 (4.713),3.000
3,50,"C1(MR=2, RT=SES, EVST=TTB, EVSP=LL, OVST=TTB, ...",11.629 (7.532),8.000
4,50,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",3.518 (6.071),5.367
5,50,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, O...",1.877 (1.103),4.333
6,50,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",2.056 (1.283),5.400
7,50,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",1.991 (1.036),4.867
8,50,"C1(MR=6, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",8.89 (2.399),8.067
9,50,"C1(MR=9, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, O...",17.79 (3.655),10.267


In [12]:
avgs_instance = group_alt_results(
    best_elite_stats,
    ["Instance type", "Server capacity", "Configuration"],
    sort_by=["Instance type", "Server capacity", "Configuration"],
    ascending=[False, True, True],
)
display(avgs_instance)
to_latex_table(avgs_instance, filename="alt_results.tex")

,Instance type,Server capacity,Configuration,Mean RPD (SD),Mean Rank
88,small,50,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",3.792 (7.622),4.1
89,small,50,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",3.868 (8.053),4.4
90,small,50,"C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",3.636 (7.947),3.0
91,small,50,"C1(MR=2, RT=SES, EVST=TTB, EVSP=LL, OVST=TTB, ...",4.73 (3.806),5.8
92,small,50,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, ...",7.469 (9.329),8.2
...,...,...,...,...,...
39,big,1000,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.531 (0.469),3.4
40,big,1000,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.684 (0.545),4.0
41,big,1000,"C1(MR=6, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.988 (0.655),4.8
42,big,1000,"C1(MR=9, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, O...",2.481 (1.272),8.3


In [13]:
# The best configuration in mean RPD
win_config_1 = "C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, OVSP=RND, NVSP=HL, PSP=RND)"
# The best configuration in mean rank
win_config_2 = "C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, OVSP=RR, NVSP=HL, PSP=RR)"

best_elite_runs_winners = best_elite_runs[
    best_elite_runs["Configuration"].isin([win_config_1, win_config_2])
]
best_elite_stats_winners = calc_stats_elites(best_elite_runs_winners)
display(
    best_elite_stats_winners.sort_values(
        by=["Instance type", "Server capacity", "Instance"],
        ascending=[False, True, True],
    )
)

,Instance type,Server capacity,Instance,Configuration,Cost,Rank,RPD
369,small,50,10t,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",3.275319e+06,2.0,0.675
409,small,50,10t,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",3.253362e+06,1.0,0.000
360,small,50,1t,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",6.111691e+06,1.0,0.000
400,small,50,1t,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",6.137934e+06,2.0,0.429
361,small,50,2t,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",3.753368e+06,1.0,0.000
...,...,...,...,...,...,...,...
1316,big,1000,7t,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",2.289303e+07,1.0,0.000
1277,big,1000,8t,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",1.929128e+07,2.0,0.301
1317,big,1000,8t,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",1.923345e+07,1.0,0.000
1278,big,1000,9t,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",2.181769e+07,2.0,0.315


In [14]:
avgs_all = group_alt_results(
    best_elite_stats_winners,
    ["Configuration"],
    sort_by=["Configuration"],
    ascending=[True],
)
display(avgs_all)
to_latex_table(avgs_all, filename="alt_results_winners_all.tex")
avgs_instance = group_alt_results(
    best_elite_stats_winners,
    ["Instance type", "Configuration"],
    sort_by=["Instance type", "Configuration"],
    ascending=[False, True],
)
to_latex_table(avgs_instance, filename="alt_results_winners_instance.tex")
display(avgs_instance)
avgs_server = group_alt_results(
    best_elite_stats_winners,
    ["Server capacity", "Configuration"],
    sort_by=["Server capacity", "Configuration"],
    ascending=[True, True],
)
to_latex_table(avgs_server, filename="alt_results_winners_server.tex")
display(avgs_server)
avgs = group_alt_results(
    best_elite_stats_winners,
    ["Instance type", "Server capacity", "Configuration"],
    sort_by=["Instance type", "Server capacity", "Configuration"],
    ascending=[False, True, True],
)
display(avgs)
to_latex_table(avgs, filename="alt_results_winners.tex")

,Configuration,Mean RPD (SD),Mean Rank
0,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.206 (0.435),1.517
1,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.433 (1.154),1.483


,Instance type,Configuration,Mean RPD (SD),Mean Rank
4,small,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.378 (0.65),1.400
5,small,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",1.001 (1.842),1.600
2,medium,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.132 (0.31),1.475
3,medium,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.181 (0.334),1.525
0,big,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.108 (0.119),1.675
1,big,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.118 (0.245),1.325


,Server capacity,Configuration,Mean RPD (SD),Mean Rank
0,50,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.18 (0.333),1.633
1,50,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.119 (0.281),1.367
2,150,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.143 (0.199),1.567
3,150,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.309 (0.711),1.433
4,650,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.294 (0.59),1.533
5,650,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.4 (0.836),1.467
6,1000,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.206 (0.516),1.333
7,1000,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.905 (1.957),1.667


,Instance type,Server capacity,Configuration,Mean RPD (SD),Mean Rank
16,small,50,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.342 (0.539),1.4
17,small,50,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.336 (0.417),1.6
18,small,150,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.123 (0.27),1.3
19,small,150,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.832 (1.074),1.7
20,small,650,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.708 (0.897),1.6
21,small,650,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.766 (1.346),1.4
22,small,1000,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.338 (0.683),1.3
23,small,1000,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",2.068 (3.116),1.7
8,medium,50,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.084 (0.117),1.6
9,medium,50,"C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, ...",0.016 (0.03),1.4


In [15]:
win_config_b = "B1(MR=2, VAdtp=TTB, VAms=HL, OMSdtp=TTB, OMSms=HL)"
win_config_a = "A1(SP=HL)"

best_elite_stats_winners_c = best_elite_runs[best_elite_runs["Configuration"].isin([win_config_1])]
best_elite_stats_winners_b = best_elite_runs_b[best_elite_runs_b["Configuration"].isin([win_config_b])]
best_elite_stats_winners_a = best_elite_runs_a[best_elite_runs_a["Configuration"].isin([win_config_a])]

best_elite_stats_winners_all = pd.concat(
    [best_elite_stats_winners_c, best_elite_stats_winners_b, best_elite_stats_winners_a],
    ignore_index=True
)
best_elite_stats_winners_all = best_elite_stats_winners_all.drop(columns=["Rank", "RPD"], errors="ignore")
best_elite_stats_winners_all["Configuration"] = best_elite_stats_winners_all["Configuration"].replace(
    {
        win_config_1: "Best C",
        win_config_b: "Best B",
        win_config_a: "Best A"
    }
)
# Instead of converting "Cost" to numeric and coercing errors to NaN, keep "E" as is and only convert numeric values.
def safe_to_numeric(val):
    try:
        return float(val)
    except (ValueError, TypeError):
        if val == "E":
            return "Error"  # keep "E" as is
        return val  # keep any non-numeric value

best_elite_stats_winners_all["Cost"] = best_elite_stats_winners_all["Cost"].apply(safe_to_numeric)
def _replace_t(row):
    inst = row.get("Instance")
    if pd.isna(inst):
        return inst
    inst = str(inst)
    if not inst.endswith("t"):
        return inst
    suffix = {"small": "s", "medium": "m", "big": "b"}.get(row.get("Instance type"), "")
    return inst[:-1] + suffix if suffix else inst

best_elite_stats_winners_all["Instance"] = best_elite_stats_winners_all.apply(_replace_t, axis=1)
best_elite_stats_winners_all = best_elite_stats_winners_all.rename(columns={"Instance": "Instance ID"})

# Pivot, but use aggfunc='first' to preserve "E" if present
best_elite_stats_winners_all_pivot = best_elite_stats_winners_all.pivot_table(
    index=["Instance type", "Server capacity", "Instance ID"],
    columns="Configuration",
    values="Cost",
    aggfunc="first"
).reset_index().sort_values(by=["Instance type", "Server capacity", "Instance ID"], ascending=[False, True, True])

def add_pct_diff(df):

    result = df.copy()
    result["Best A % Diff"] = None
    result["Best B % Diff"] = None

    for idx, row in df.iterrows():
        best_c = row.get("Best C")
        for col, pct_col in [("Best A", "Best A % Diff"), ("Best B", "Best B % Diff")]:
            val = row.get(col)
            if isinstance(val, (int, float, np.floating)) and isinstance(best_c, (int, float, np.floating)):
                diff = 100 * abs((val - best_c) / best_c)
                result.at[idx, pct_col] = diff
            else:
                # propagate 'Error' or leave None
                if val == "Error" or best_c == "Error":
                    result.at[idx, pct_col] = "Error"
                else:
                    result.at[idx, pct_col] = None
    return result

beswap_with_diff = add_pct_diff(best_elite_stats_winners_all_pivot)
display(beswap_with_diff)

Configuration,Instance type,Server capacity,Instance ID,Best A,Best B,Best C,Best A % Diff,Best B % Diff
80,small,50,10s,Error,3224462.783,3253361.635,Error,0.888277
81,small,50,1s,Error,6121071.601,6137933.771,Error,0.274721
82,small,50,2s,Error,3369278.519,3814906.093,Error,11.68122
83,small,50,3s,Error,3404735.609,3668356.062,Error,7.186338
84,small,50,4s,Error,7370408.85,7551668.512,Error,2.40026
...,...,...,...,...,...,...,...,...
35,big,1000,5b,15421104.102,13461380.949,11725715.03,31.515256,14.802218
36,big,1000,6b,34428240.111,30240915.876,26649780.997,29.187704,13.475289
37,big,1000,7b,26048557.485,22751588.416,22893029.008,13.783796,0.617833
38,big,1000,8b,25752812.767,22559925.699,19233447.106,33.895981,17.29528


In [16]:
# select only the needed cols
grouped = beswap_with_diff[
    ["Instance type", "Best A % Diff", "Best B % Diff"]
].copy()

def replace_error(group):
# convert "Error" → NaN and cast to numeric
    group[["Best A % Diff", "Best B % Diff"]] = (
        group[["Best A % Diff", "Best B % Diff"]]
        .replace("Error", np.nan)
        .apply(pd.to_numeric, errors="coerce")
    )
replace_error(grouped)
# group and compute mean (skipna=True by default), then round
grouped_instance = grouped.groupby(["Instance type"]).mean().round(3).reset_index()
display(grouped_instance)
to_latex_table(grouped_instance, filename="alg_comparison_instance.tex")

Configuration,Instance type,Best A % Diff,Best B % Diff
0,big,23.830,8.050
1,medium,39.260,15.860
2,small,45.043,10.008


In [17]:
# group and compute mean (skipna=True by default), then round
grouped = beswap_with_diff[
    ["Server capacity", "Best A % Diff", "Best B % Diff"]
].copy()
replace_error(grouped)
grouped = grouped.groupby(["Server capacity"]).mean().round(3).reset_index()

display(grouped)
to_latex_table(grouped, filename="alg_comparison_server.tex")

Configuration,Server capacity,Best A % Diff,Best B % Diff
0,50,NaN,4.765
1,150,12.863,8.609
2,650,37.022,13.412
3,1000,54.592,18.439


In [18]:
# group and compute mean (skipna=True by default), then round
grouped = beswap_with_diff[
    ["Best A % Diff", "Best B % Diff"]
].copy()
replace_error(grouped)
mean_vals = grouped.mean().round(3)

display(mean_vals)

Configuration
Best A % Diff    37.571
Best B % Diff    11.306
dtype: float64

In [19]:
base_cost = 0.005 / 3600 # 0.005 USD per hour in seconds
mults = {
    50: 1,
    150: 3,
    650: 13,
    1000: 20
}
exps = {
    50: 1, # base, r=1.25
    150: 3 * 1.25,
    650: 13 * 1.25,
    1000: 20 * 1.25
}
costs_df = best_elite_stats_winners_all_pivot[["Instance type", "Server capacity", "Instance ID", "Best C"]].copy()
costs_df = costs_df[
    costs_df["Instance ID"].str.contains("1", na=False)
    & ~costs_df["Instance ID"].str.contains("0", na=False)
].copy()

# Safely convert Best C to numeric (non-numeric -> NaN)
costs_df["Best C (numeric)"] = pd.to_numeric(costs_df["Best C"], errors="coerce")
costs_df_exp = costs_df.copy()

# Map server capacity to multiplier
costs_df["Multiplier"] = costs_df["Server capacity"].map(mults)
costs_df_exp["Multiplier"] = costs_df_exp["Server capacity"].map(exps)

# Compute USD cost only for numeric Best C and available multiplier.
# Preserve "Error" (or "E") labels if present in the original Best C.
def compute_cost(row):
    bc = row["Best C (numeric)"]
    mult = row["Multiplier"]
    orig = row["Best C"]
    if pd.notna(bc) and pd.notna(mult):
        return round(bc * base_cost * mult, 2)
    if orig in ("Error", "E"):
        return "Error"
    return pd.NA

costs_df["Cost per server (USD/server)"] = round(base_cost * 3600 * costs_df["Multiplier"], 5)
costs_df_exp["Cost per server (USD/server)"] = round(base_cost * 3600 *costs_df_exp["Multiplier"], 5)

costs_df["Total cost (USD)"] = costs_df.apply(compute_cost, axis=1)
costs_df_exp["Total cost (USD)"] = costs_df_exp.apply(compute_cost, axis=1)

# cleanup helper columns
costs_df = costs_df.drop(columns=["Best C (numeric)", "Instance ID", "Best C", "Multiplier"])
costs_df_exp = costs_df_exp.drop(columns=["Best C (numeric)", "Instance ID", "Best C", "Multiplier"])

display(costs_df)
display(costs_df_exp)
to_latex_table(costs_df, filename="costs.tex")
to_latex_table(costs_df_exp, filename="costs_exp.tex")

Configuration,Instance type,Server capacity,Cost per server (USD/server),Total cost (USD)
81,small,50,0.005,8.52
91,small,150,0.015,8.10
101,small,650,0.065,9.50
111,small,1000,0.100,9.31
41,medium,50,0.005,193.29
51,medium,150,0.015,182.40
61,medium,650,0.065,185.42
71,medium,1000,0.100,188.05
1,big,50,0.005,713.50
11,big,150,0.015,675.81


Configuration,Instance type,Server capacity,Cost per server (USD/server),Total cost (USD)
81,small,50,0.00500,8.52
91,small,150,0.01875,10.12
101,small,650,0.08125,11.87
111,small,1000,0.12500,11.64
41,medium,50,0.00500,193.29
51,medium,150,0.01875,227.99
61,medium,650,0.08125,231.77
71,medium,1000,0.12500,235.06
1,big,50,0.00500,713.50
11,big,150,0.01875,844.77
